In [1]:
"""
Q-G-CTGAN: Quality-Aware Cluster-Conditioned Oversampling
via Intra-Cluster Synthetic Sample Filtering

Notebook 02b: Non-Generative Baseline Methods (Part B: Large Datasets)

This notebook evaluates the same four non-generative oversampling
baselines (None, SMOTE, ADASYN, G-SMOTE) on the three largest benchmark
datasets (fraud_detection, unsw_nb15, protein_homo), which were excluded
from 02a_baselines_small.ipynb due to their size and the poor scaling
behavior of k-NN-based oversamplers with sample count.

NOTE: The oversampler label "None" is renamed here to "NoOverSampling"
to avoid a pandas quirk where the literal string "None" is silently
interpreted as a missing value on CSV read-back (observed in the
02a results and worked around there with keep_default_na=False;
here we avoid the issue at the source instead).

Results are saved incrementally after each (dataset, oversampler)
combination completes (finer granularity than 02a), since a single
combination can itself take several minutes on these datasets.
"""

import os
import time
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier

from imblearn.over_sampling import SMOTE, ADASYN
from imblearn_extra.gsmote import GeometricSMOTE

warnings.filterwarnings("ignore")

# -- Paths --------------------------------------------------
DATASET_DIR = "./datasets"
RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# -- Reproducibility ------------------------------------------
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# -- This run: large datasets only ------------------------------
DATASET_NAMES = ["protein_homo", "unsw_nb15", "fraud_detection"]  # smallest to largest

print(f"Dataset directory : {DATASET_DIR}")
print(f"Results directory : {RESULTS_DIR}")
print(f"Random state      : {RANDOM_STATE}")
print(f"Datasets in this run: {DATASET_NAMES}")


## 1. Dataset Registry

datasets = {}
for name in DATASET_NAMES:
    path = os.path.join(DATASET_DIR, f"{name}.csv")
    df = pd.read_csv(path)
    datasets[name] = df
    minority = int(df["target"].sum())
    ir = round((len(df) - minority) / minority, 2)
    n_bool_cols = (df.dtypes == "bool").sum()
    print(f"  {name:<20}  n={len(df):>7,}  features={df.shape[1]-1:>4}  "
          f"IR={ir:>7.1f}  bool_cols={n_bool_cols}")


## 2. Classifier Factory

def get_classifiers(random_state=RANDOM_STATE):
    return {
        "RF": RandomForestClassifier(
            n_estimators=200, max_depth=10, min_samples_leaf=3,
            class_weight="balanced", random_state=random_state, n_jobs=-1,
        ),
        "LGBM": LGBMClassifier(
            n_estimators=100, learning_rate=0.05, num_leaves=31,
            random_state=random_state, verbosity=-1, n_jobs=-1,
        ),
        "MLP": MLPClassifier(
            hidden_layer_sizes=(128, 64), activation="relu",
            alpha=0.001, random_state=random_state, max_iter=500,
        ),
    }


## 3. Oversampler Factory
# "NoOverSampling" replaces "None" as the label for the no-oversampling
# baseline (see module docstring for rationale).

def get_oversamplers(random_state=RANDOM_STATE):
    return {
        "NoOverSampling": None,
        "SMOTE": SMOTE(random_state=random_state),
        "ADASYN": ADASYN(random_state=random_state),
        "G-SMOTE": GeometricSMOTE(random_state=random_state),
    }


## 4. Evaluation Loop
# Same protocol as 02a (stratified 5-fold CV, oversampling applied only
# to the training fold), but with per-(dataset, oversampler) incremental
# saving, since a single combination can take several minutes here.

N_FOLDS = 5

results = []
run_start = time.time()
out_path = os.path.join(RESULTS_DIR, "02b_baseline_results_large.csv")

for ds_name, df in datasets.items():
    X = df.drop(columns=["target"]).values.astype(np.float64)
    y = df["target"].values

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    for sampler_name, sampler in get_oversamplers().items():
        combo_start = time.time()

        for clf_name, clf in get_classifiers().items():

            fold_aucs = []
            fold_times = []

            for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y)):
                X_train, X_test = X[train_idx], X[test_idx]
                y_train, y_test = y[train_idx], y[test_idx]

                t0 = time.time()

                if sampler is not None:
                    try:
                        X_train_res, y_train_res = sampler.fit_resample(X_train, y_train)
                    except Exception as e:
                        print(f"  [WARN] {ds_name}/{sampler_name} fold {fold_idx}: "
                              f"resample failed ({e}); using original training data")
                        X_train_res, y_train_res = X_train, y_train
                else:
                    X_train_res, y_train_res = X_train, y_train

                clf_fold = get_classifiers()[clf_name]
                clf_fold.fit(X_train_res, y_train_res)
                y_prob = clf_fold.predict_proba(X_test)[:, 1]

                auc = roc_auc_score(y_test, y_prob)
                elapsed = time.time() - t0

                fold_aucs.append(auc)
                fold_times.append(elapsed)

            results.append({
                "dataset": ds_name,
                "oversampler": sampler_name,
                "classifier": clf_name,
                "auc_mean": np.mean(fold_aucs),
                "auc_std": np.std(fold_aucs),
                "time_mean_sec": np.mean(fold_times),
                "time_total_sec": np.sum(fold_times),
            })

            print(f"  [{ds_name:<15}] {sampler_name:<15} + {clf_name:<5}  "
                  f"AUC={np.mean(fold_aucs):.4f}+-{np.std(fold_aucs):.4f}  "
                  f"time={np.sum(fold_times):.1f}s")

        # Incremental save after each (dataset, oversampler) combination
        pd.DataFrame(results).to_csv(out_path, index=False)
        combo_elapsed = time.time() - combo_start
        print(f"  --- {ds_name}/{sampler_name} complete in {combo_elapsed:.1f}s "
              f"(saved to {out_path}) ---\n")

run_elapsed = time.time() - run_start

results_df = pd.DataFrame(results)
results_df.to_csv(out_path, index=False)

print(f"\nTotal wall-clock time: {run_elapsed:.1f}s ({run_elapsed/60:.1f} min)")
print(f"Saved {len(results_df)} result rows to {out_path}")

Dataset directory : ./datasets
Results directory : ./results
Random state      : 42
Datasets in this run: ['protein_homo', 'unsw_nb15', 'fraud_detection']
  protein_homo          n=145,751  features=  74  IR=  111.5  bool_cols=0
  unsw_nb15             n=175,341  features=  34  IR=   99.4  bool_cols=0
  fraud_detection       n=284,807  features=  30  IR=  577.9  bool_cols=0
  [protein_homo   ] NoOverSampling  + RF     AUC=0.9923+-0.0015  time=28.9s
  [protein_homo   ] NoOverSampling  + LGBM   AUC=0.9935+-0.0012  time=3.3s
  [protein_homo   ] NoOverSampling  + MLP    AUC=0.9779+-0.0078  time=228.0s
  --- protein_homo/NoOverSampling complete in 260.6s (saved to ./results\02b_baseline_results_large.csv) ---

  [protein_homo   ] SMOTE           + RF     AUC=0.9922+-0.0011  time=112.3s
  [protein_homo   ] SMOTE           + LGBM   AUC=0.9927+-0.0006  time=6.9s
  [protein_homo   ] SMOTE           + MLP    AUC=0.9806+-0.0021  time=494.2s
  --- protein_homo/SMOTE complete in 614.0s (saved to ./

ValueError: could not convert string to float: 'tcp'

In [5]:
### 1-10 (fixed). Binarize UNSW-NB15 - Backdoor detection (object + str dtype)

df_unsw_raw = pd.read_parquet("./datasets/raw_unsw/UNSW_NB15_training-set.parquet")

df = df_unsw_raw.copy()
df["target"] = (df["attack_cat"] == "Backdoor").astype(int)
df = df.drop(columns=["attack_cat", "label"])

cat_cols = df.select_dtypes(include=["object", "str"]).columns
print(f"Categorical columns detected: {list(cat_cols)}")

if len(cat_cols):
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

minority = int(df["target"].sum())
majority = len(df) - minority
ir = round(majority / minority, 2)

path = os.path.join("./datasets", "unsw_nb15.csv")
df.to_csv(path, index=False)

print(f"  [OK] {'UNSW-NB15':<20}  n={len(df):>7,}  minority={minority:>5,}  IR={ir:>7.1f}  features={df.shape[1]-1}  -> saved (re-encoded)")

Categorical columns detected: []
  [OK] UNSW-NB15             n=175,341  minority=1,746  IR=   99.4  features=34  -> saved (re-encoded)


In [9]:
# Resume 02b: skip protein_homo (already complete), start from unsw_nb15
DATASET_NAMES = ["unsw_nb15", "fraud_detection"]

datasets = {}
for name in DATASET_NAMES:
    path = os.path.join(DATASET_DIR, f"{name}.csv")
    df = pd.read_csv(path)
    datasets[name] = df
    minority = int(df["target"].sum())
    ir = round((len(df) - minority) / minority, 2)
    n_bool_cols = (df.dtypes == "bool").sum()
    print(f"  {name:<20}  n={len(df):>7,}  features={df.shape[1]-1:>4}  "
          f"IR={ir:>7.1f}  bool_cols={n_bool_cols}")

  unsw_nb15             n=175,341  features= 183  IR=   99.4  bool_cols=152
  fraud_detection       n=284,807  features=  30  IR=  577.9  bool_cols=0


In [10]:
## 4. Evaluation Loop (resumed: unsw_nb15, fraud_detection only)
# protein_homo results (already computed and saved) are loaded and
# prepended, so the final CSV contains all three large datasets.

N_FOLDS = 5

out_path = os.path.join(RESULTS_DIR, "02b_baseline_results_large.csv")

# Load previously completed protein_homo results
existing_results = pd.read_csv(out_path, keep_default_na=False)
results = existing_results.to_dict("records")
print(f"Loaded {len(results)} existing result rows (protein_homo)")

run_start = time.time()

for ds_name, df in datasets.items():
    X = df.drop(columns=["target"]).values.astype(np.float64)
    y = df["target"].values

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    for sampler_name, sampler in get_oversamplers().items():
        combo_start = time.time()

        for clf_name, clf in get_classifiers().items():

            fold_aucs = []
            fold_times = []

            for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y)):
                X_train, X_test = X[train_idx], X[test_idx]
                y_train, y_test = y[train_idx], y[test_idx]

                t0 = time.time()

                if sampler is not None:
                    try:
                        X_train_res, y_train_res = sampler.fit_resample(X_train, y_train)
                    except Exception as e:
                        print(f"  [WARN] {ds_name}/{sampler_name} fold {fold_idx}: "
                              f"resample failed ({e}); using original training data")
                        X_train_res, y_train_res = X_train, y_train
                else:
                    X_train_res, y_train_res = X_train, y_train

                clf_fold = get_classifiers()[clf_name]
                clf_fold.fit(X_train_res, y_train_res)
                y_prob = clf_fold.predict_proba(X_test)[:, 1]

                auc = roc_auc_score(y_test, y_prob)
                elapsed = time.time() - t0

                fold_aucs.append(auc)
                fold_times.append(elapsed)

            results.append({
                "dataset": ds_name,
                "oversampler": sampler_name,
                "classifier": clf_name,
                "auc_mean": np.mean(fold_aucs),
                "auc_std": np.std(fold_aucs),
                "time_mean_sec": np.mean(fold_times),
                "time_total_sec": np.sum(fold_times),
            })

            print(f"  [{ds_name:<15}] {sampler_name:<15} + {clf_name:<5}  "
                  f"AUC={np.mean(fold_aucs):.4f}+-{np.std(fold_aucs):.4f}  "
                  f"time={np.sum(fold_times):.1f}s")

        pd.DataFrame(results).to_csv(out_path, index=False)
        combo_elapsed = time.time() - combo_start
        print(f"  --- {ds_name}/{sampler_name} complete in {combo_elapsed:.1f}s "
              f"(saved to {out_path}) ---\n")

run_elapsed = time.time() - run_start

results_df = pd.DataFrame(results)
results_df.to_csv(out_path, index=False)

print(f"\nTotal wall-clock time this session: {run_elapsed:.1f}s ({run_elapsed/60:.1f} min)")
print(f"Total rows in final file: {len(results_df)}")

Loaded 12 existing result rows (protein_homo)
  [unsw_nb15      ] NoOverSampling  + RF     AUC=0.9062+-0.0016  time=20.6s
  [unsw_nb15      ] NoOverSampling  + LGBM   AUC=0.9176+-0.0022  time=2.5s
  [unsw_nb15      ] NoOverSampling  + MLP    AUC=0.5085+-0.0074  time=652.0s
  --- unsw_nb15/NoOverSampling complete in 676.6s (saved to ./results\02b_baseline_results_large.csv) ---

  [unsw_nb15      ] SMOTE           + RF     AUC=0.9113+-0.0036  time=58.1s
  [unsw_nb15      ] SMOTE           + LGBM   AUC=0.9172+-0.0046  time=8.3s
  [unsw_nb15      ] SMOTE           + MLP    AUC=0.5085+-0.0008  time=1045.4s
  --- unsw_nb15/SMOTE complete in 1113.5s (saved to ./results\02b_baseline_results_large.csv) ---

  [unsw_nb15      ] ADASYN          + RF     AUC=0.9141+-0.0038  time=55.7s
  [unsw_nb15      ] ADASYN          + LGBM   AUC=0.9164+-0.0051  time=9.3s
  [unsw_nb15      ] ADASYN          + MLP    AUC=0.5086+-0.0013  time=1136.6s
  --- unsw_nb15/ADASYN complete in 1203.3s (saved to ./results